# Task_A_alignment: Alignment using PPO

In [1]:
# Importing the required libraries
import os
import gc
import torch
import numpy as np
import random
from datasets import load_dataset
from transformers import AutoTokenizer, pipeline, AutoModelForSequenceClassification, AutoModelForCausalLM, BitsAndBytesConfig
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer, create_reference_model
from peft import TaskType, LoraConfig, PrefixTuningConfig, PeftConfig, PeftModel, get_peft_model
from huggingface_hub import login
from dotenv import load_dotenv
from tqdm import tqdm
import warnings

# Seed Setup
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [2]:
# Environment Setup
warnings.filterwarnings('ignore')
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# HF login
load_dotenv()
login(token=os.getenv("HF_TOKEN"))
hf_username = "sohammandal01"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# Loading the base model
base_model_id = "gpt2-medium"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token



In [4]:
# Loading and preparing dataset
print("Loading and preparing dataset...")
dataset = load_dataset("nrizwan/safe_ai_assignment_1", data_files={"train": "preference_train.csv"})
train_dataset = dataset['train'].shuffle(seed=42).select(range(4000))


def build_dataset(batch):
    queries = []
    input_ids = []
    for q in batch['Question']:
        query = f"Question: {q}\nAnswer:"
        tokenized_query = tokenizer(query, truncation=True, max_length=128)["input_ids"]
        queries.append(query)
        input_ids.append(tokenized_query)
    return {"query": queries, "input_ids": input_ids}

ppo_dataset = train_dataset.map(build_dataset, batched=True, batch_size=100)
ppo_dataset.set_format(type="torch", columns=["input_ids", "query"])

def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])

generation_kwargs = {
    "min_length": -1,
    "top_k": 3,
    "top_p": 0.7,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
    "max_new_tokens": 30,
}
print("Dataset and collator ready!")

Loading and preparing dataset...
Dataset and collator ready!


## Full Parameter Fine-Tuning

In [8]:
print("Starting Full Parameter PPO Alignment...")
torch.cuda.empty_cache()
gc.collect()

reward_model_id = f"{hf_username}/bert-reward-model-assignment"
reward_pipe = pipeline("text-classification", model=reward_model_id, tokenizer=reward_model_id, device=0)

model_full = AutoModelForCausalLMWithValueHead.from_pretrained(base_model_id, device_map={"": 0})
ref_model_full = create_reference_model(model_full)

ppo_config_full = PPOConfig(
    batch_size=4,
    mini_batch_size=1,
    learning_rate=1e-6,
    init_kl_coef=0.5,
    cliprange=0.1,
    steps=1000,
    optimize_cuda_cache=True
)

ppo_trainer_full = PPOTrainer(config=ppo_config_full, model=model_full, ref_model=ref_model_full,tokenizer=tokenizer, dataset=ppo_dataset, data_collator=collator)

for epoch, batch in tqdm(enumerate(ppo_trainer_full.dataloader)):
    query_tensors = batch["input_ids"]
    response_tensors = ppo_trainer_full.generate(query_tensors, **generation_kwargs)
    response_tensors = [r[len(q):] for q, r in zip(query_tensors, response_tensors)]
    batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

    texts_to_score = [q + r for q, r in zip(batch["query"], batch["response"])]
    pipe_outputs = reward_pipe(texts_to_score)
    rewards = [torch.tensor(output["score"]) for output in pipe_outputs]

    stats = ppo_trainer_full.step(query_tensors, response_tensors, rewards)

    if epoch % 50 == 0:
        print(f"Step {epoch}: Average Reward = {torch.mean(torch.stack(rewards)).item():.4f}")

repo_full = f"{hf_username}/gpt2-aligned-ppo-full-assignment"
ppo_trainer_full.model.push_to_hub(repo_full)
tokenizer.push_to_hub(repo_full)
print(f"Full PPO model pushed to: {repo_full}")

Starting Full Parameter PPO Alignment...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

1it [00:02,  2.67s/it]

Step 0: Average Reward = 0.0017


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Full PPO model pushed to: sohammandal01/gpt2-aligned-ppo-full-assignment


# Prefix Tuning

In [9]:
print("Starting Prefix Tuning PPO Alignment...")
torch.cuda.empty_cache()
gc.collect()

reward_model_id = f"{hf_username}/bert-reward-model-assignment"
reward_pipe = pipeline("text-classification", model=reward_model_id, tokenizer=reward_model_id, device=0)

base_lm = AutoModelForCausalLM.from_pretrained(base_model_id, device_map={"": 0})
peft_config_prefix = PrefixTuningConfig(task_type=TaskType.CAUSAL_LM, num_virtual_tokens=20)
peft_model_prefix = get_peft_model(base_lm, peft_config_prefix)

model_prefix = AutoModelForCausalLMWithValueHead(peft_model_prefix)
model_prefix.is_peft_model = True
if hasattr(model_prefix, 'pretrained_model'):
    model_prefix.pretrained_model.is_peft_model = True

ppo_config_prefix = PPOConfig(
    batch_size=8,
    mini_batch_size=2,
    learning_rate=1e-3,
    steps=500,
    optimize_cuda_cache=True
)

ppo_trainer_prefix = PPOTrainer(config=ppo_config_prefix, model=model_prefix, ref_model=None,tokenizer=tokenizer, dataset=ppo_dataset, data_collator=collator)

print("Starting PPO step loop...")
for epoch, batch in tqdm(enumerate(ppo_trainer_prefix.dataloader)):
  query_tensors = batch["input_ids"]
  response_tensors = ppo_trainer_prefix.generate(query_tensors, **generation_kwargs)
  response_tensors = [r[len(q):] for q, r in zip(query_tensors, response_tensors)]
  batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

  texts_to_score = [q + r for q, r in zip(batch["query"], batch["response"])]
  pipe_outputs = reward_pipe(texts_to_score)
  rewards = [torch.tensor(output["score"]) for output in pipe_outputs]

  stats = ppo_trainer_prefix.step(query_tensors, response_tensors, rewards)

  if epoch % 50 == 0:
    print(f"Step {epoch}: Average Reward = {torch.mean(torch.stack(rewards)).item():.4f}")

repo_prefix = f"{hf_username}/gpt2-aligned-ppo-prefix-assignment"
ppo_trainer_prefix.model.push_to_hub(repo_prefix)
tokenizer.push_to_hub(repo_prefix)
print(f"Prefix PPO model pushed to: {repo_prefix}")

Starting Prefix Tuning PPO Alignment...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Starting PPO step loop...


1it [00:02,  2.72s/it]

...

500it [24:47,  2.98s/it]


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Prefix PPO model pushed to: sohammandal01/gpt2-aligned-ppo-prefix-assignment


# LoRA

In [5]:
print("Starting LoRA PPO Alignment...")
torch.cuda.empty_cache()
gc.collect()

reward_model_id = f"{hf_username}/bert-reward-model-assignment"
reward_pipe = pipeline("text-classification", model=reward_model_id, tokenizer=reward_model_id, device=0)

base_lm = AutoModelForCausalLM.from_pretrained(base_model_id, device_map={"": 0})
peft_config_lora = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=32, lora_dropout=0.1,target_modules=["c_attn"])
peft_model_lora = get_peft_model(base_lm, peft_config_lora)

model_lora = AutoModelForCausalLMWithValueHead(peft_model_lora)
model_lora.is_peft_model = True
if hasattr(model_lora, 'pretrained_model'):
    model_lora.pretrained_model.is_peft_model = True

ppo_config_lora = PPOConfig(batch_size=8, mini_batch_size=2, learning_rate=1.41e-4, steps=500, optimize_cuda_cache=True)
ppo_trainer_lora = PPOTrainer(config=ppo_config_lora, model=model_lora, ref_model=None,tokenizer=tokenizer, dataset=ppo_dataset, data_collator=collator)

for epoch, batch in tqdm(enumerate(ppo_trainer_lora.dataloader)):
    query_tensors = batch["input_ids"]
    response_tensors = ppo_trainer_lora.generate(query_tensors, **generation_kwargs)
    response_tensors = [r[len(q):] for q, r in zip(query_tensors, response_tensors)]
    batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

    texts_to_score = [q + r for q, r in zip(batch["query"], batch["response"])]
    pipe_outputs = reward_pipe(texts_to_score)
    rewards = [torch.tensor(output["score"]) for output in pipe_outputs]

    stats = ppo_trainer_lora.step(query_tensors, response_tensors, rewards)

    if epoch % 50 == 0:
        print(f"Step {epoch}: Average Reward = {torch.mean(torch.stack(rewards)).item():.4f}")

repo_lora = f"{hf_username}/gpt2-aligned-ppo-lora-assignment"
ppo_trainer_lora.model.push_to_hub(repo_lora)
tokenizer.push_to_hub(repo_lora)
print(f"LoRA PPO model pushed to: {repo_lora}")

Starting LoRA PPO Alignment...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

1it [00:03,  3.13s/it]

Step 0: Average Reward = 0.0461


...

Step 450: Average Reward = 0.0399


500it [26:11,  3.14s/it]


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

LoRA PPO model pushed to: sohammandal01/gpt2-aligned-ppo-lora-assignment


# QLoRA

In [ ]:
print("Starting QLoRA PPO Alignment...")
torch.cuda.empty_cache()
gc.collect()

reward_model_id = f"{hf_username}/bert-reward-model-assignment"
reward_pipe = pipeline("text-classification", model=reward_model_id, tokenizer=reward_model_id, device=0)

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
base_lm_qlora = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config, device_map={"": 0})
peft_config_qlora = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=32, lora_dropout=0.1,target_modules=["c_attn"])

peft_model_qlora = get_peft_model(base_lm_qlora, peft_config_qlora)

model_qlora = AutoModelForCausalLMWithValueHead(peft_model_qlora)
model_qlora.is_peft_model = True
if hasattr(model_qlora, 'pretrained_model'):
    model_qlora.pretrained_model.is_peft_model = True

ppo_config_qlora = PPOConfig(batch_size=8, mini_batch_size=2, learning_rate=1.41e-4, steps=500, optimize_cuda_cache=True)
ppo_trainer_qlora = PPOTrainer(config=ppo_config_qlora, model=model_qlora, ref_model=None,tokenizer=tokenizer, dataset=ppo_dataset, data_collator=collator)

for epoch, batch in tqdm(enumerate(ppo_trainer_qlora.dataloader)):
    query_tensors = batch["input_ids"]
    response_tensors = ppo_trainer_qlora.generate(query_tensors, **generation_kwargs)
    response_tensors = [r[len(q):] for q, r in zip(query_tensors, response_tensors)]
    batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

    texts_to_score = [q + r for q, r in zip(batch["query"], batch["response"])]
    pipe_outputs = reward_pipe(texts_to_score)
    rewards = [torch.tensor(output["score"]) for output in pipe_outputs]

    stats = ppo_trainer_qlora.step(query_tensors, response_tensors, rewards)

    if epoch % 50 == 0:
        print(f"Step {epoch}: Average Reward = {torch.mean(torch.stack(rewards)).item():.4f}")

repo_qlora = f"{hf_username}/gpt2-aligned-ppo-qlora-assignment"
ppo_trainer_qlora.model.push_to_hub(repo_qlora)
tokenizer.push_to_hub(repo_qlora)
print(f"QLoRA PPO model pushed to: {repo_qlora}")

Starting QLoRA PPO Alignment...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
1it [00:08,  8.66s/it]

Step 0: Average Reward = 0.0329


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  17%|#7        |  537kB / 3.15MB            

No files have been modified since last commit. Skipping to prevent empty commit.


QLoRA PPO model pushed to: sohammandal01/gpt2-aligned-ppo-qlora-assignment


In [ ]:
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00
